# 02 · Retries, backoff, rate limits and circuit breakers

Turn a flaky upstream into a reliable service: retry only what a retry can fix, back off with jitter, honour
`Retry-After`, make side effects idempotent, rate-limit yourself with a token bucket, trip a circuit breaker and
fall back to a second model.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · Async concurrency](01_async_concurrency_for_llm_calls.ipynb)

## Why this matters in interviews

- "You are getting 429s from the provider, what is the handling?" comes up in almost every GenAI system design
  round. The answer that loses is "retry it"; the answer that wins classifies the error first.
- Retries are the classic **self-inflicted outage**: no jitter means synchronised waves that knock a recovering
  service over again, and retries stacked at three layers multiply load 27x.
- Circuit breakers and fallbacks decide whether a provider outage is *their* incident or *yours*.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `po16` | How do you design a retry policy that does not make things worse? |
| `po07` | You are getting 429s from your model provider — what is the handling? |
| `po08` | What is exponential backoff, and why is jitter not optional? |
| `in25` | How do you handle rate limits properly? |
| `po06` | How does rate limiting work, and where do you enforce it? |
| `pd12` | How do you rate limit an LLM API you own? |
| `po15` | What is a circuit breaker, and where would you put one in an LLM system? |
| `po18` | What is idempotency and why does an agent system need it more than a normal one? |
| `in28` | What is a fallback model strategy? |
| `po03` | The LLM provider has an outage. How does your system stay online? |

## Setup

Offline, the stand-in can be told to fail: `OFFLINE.fail_next(2, 429)` makes the next two requests return
HTTP 429 (with a `retry-after-ms: 100` header, like OpenAI). A real provider cannot be ordered to fail, so those
cells are guarded with `if OFFLINE:` and simply succeed with a real key. Everything else (backoff, the herd, the
token bucket, the breaker) is simulated in plain Python and runs identically everywhere.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import email.utils
import hashlib
import heapq
import random
import time
import uuid
from collections import deque

import httpx
import matplotlib.pyplot as plt
import numpy as np
import openai
from llm_setup import OFFLINE, make_client

MSG = [{"role": "user", "content": "What is 17*23?"}]
client.chat.completions.create(model=MODEL, messages=MSG, max_tokens=5)   # warm-up: the first call pays import costs
c0 = make_client(max_retries=0)       # a client with SDK retries OFF, so failures reach our code

def requests_served():
    """HTTP requests the offline stand-in has answered so far (None with a real provider)."""
    return OFFLINE.request_count if OFFLINE else None

print(f"provider={PROVIDER}, model={MODEL}")

## 1. Classify before you retry

**In plain English:** a retry only helps if something *outside your request* can change between attempts: the
quota refills, an overloaded replica recovers, a dropped connection comes back. If the request itself is wrong,
it will be exactly as wrong the second time, and you pay again.

| Failure | HTTP | `openai` exception | Retry? | Why |
|---|---|---|---|---|
| Rate limited | 429 | `RateLimitError` | **yes**, after `Retry-After` or backoff | quota refills |
| Server error / overloaded | 500, 502, 503, 504 | `InternalServerError` | **yes**, with backoff | another attempt may land on a healthy replica |
| Timeout | none | `APITimeoutError` | **yes**, once or twice, inside the deadline | may have been a slow replica |
| Connection reset, DNS | none | `APIConnectionError` | **yes** | network blip |
| Bad request (schema, context too long) | 400 | `BadRequestError` | **no** | fix the request (trim the context) |
| Auth / permission | 401, 403 | `AuthenticationError`, `PermissionDeniedError` | **no** | page someone: the key is wrong |
| Not found (wrong model name) | 404 | `NotFoundError` | **no** | config bug |
| Content policy refusal | 400 | `BadRequestError` | **no** | same input, same refusal |

In [ ]:
RETRYABLE = (openai.RateLimitError, openai.InternalServerError, openai.APIConnectionError)  # APITimeoutError is an APIConnectionError

def is_retryable(exc: BaseException) -> bool:
    """Retry only what a retry can fix: rate limits, 5xx, timeouts, dropped connections."""
    if isinstance(exc, RETRYABLE):
        return True
    if isinstance(exc, openai.APIStatusError):
        return exc.status_code in (408, 409)          # request timeout / lock conflict (the SDK retries these too)
    return False

req = httpx.Request("POST", "https://api.example.com/v1/chat/completions")
print("APITimeoutError   retry?", is_retryable(openai.APITimeoutError(request=req)))
print("APIConnectionError retry?", is_retryable(openai.APIConnectionError(request=req)))
print("ValueError (our bug) retry?", is_retryable(ValueError("bad JSON from our own parser")))

if OFFLINE:
    rows = []
    for status in (400, 401, 404, 429, 500, 503):
        OFFLINE.fail_next(1, status)
        try:
            c0.chat.completions.create(model=MODEL, messages=MSG)
        except openai.APIStatusError as e:
            rows.append((status, type(e).__name__, is_retryable(e)))
    for status, name, retry in rows:
        print(f"HTTP {status} -> openai.{name:<22} retry? {'yes' if retry else 'NO'}")
    assert [r for *_, r in rows] == [False, False, False, True, True, True]

## 2. The SDK's built-in retries

The `openai` SDK already retries. Offline we inject two 429s and time a normal call.

In [ ]:
if OFFLINE:
    OFFLINE.fail_next(2, 429)                 # the next two requests get HTTP 429
before = requests_served()
t0 = time.perf_counter()
r = client.chat.completions.create(model=MODEL, messages=MSG, max_tokens=20)    # default: max_retries=2
elapsed = time.perf_counter() - t0
print(f"answer: {r.choices[0].message.content!r} in {elapsed:.2f} s")
if OFFLINE:
    print(f"HTTP requests behind that one call: {requests_served() - before} (429, 429, then 200)")
    assert requests_served() - before == 3 and elapsed < 1.0

You never saw the two failures: the SDK waited the `retry-after-ms: 100` the server sent and tried again. With
retries switched off, the error reaches your code, carrying everything you need to decide what to do:

In [ ]:
if OFFLINE:
    OFFLINE.fail_next(1, 429)
try:
    c0.chat.completions.create(model=MODEL, messages=MSG)
    print("no error (a real provider was not told to fail)")
except openai.RateLimitError as e:
    print(f"{type(e).__name__}  status={e.status_code}")
    print(f"  retry-after-ms header : {e.response.headers.get('retry-after-ms')}")
    print(f"  request id            : {e.request_id}   <- log it; the provider's support will ask for it")
    print(f"  message               : {e.body.get('message') if isinstance(e.body, dict) else e.body}")

What the SDK does by default (read from the `openai-python` 2.x source; check yours with
`help(openai.OpenAI)`):

- retries connection errors, timeouts, **408, 409, 429 and all 5xx**, never other 4xx; `max_retries=2` by default;
- waits what the server asks (`retry-after-ms`, then `retry-after`) when that is at most 2 minutes, and does not
  retry at all if the server asks for longer; otherwise waits `0.5 s x 2^n` capped at 8 s, minus up to 25% jitter;
- times out after **600 s** by default. Set `timeout=` far lower for anything user-facing.

Override per client (`make_client(max_retries=5, timeout=20)`) or per call
(`client.with_options(max_retries=0).chat.completions.create(...)`).

## 3. Exponential backoff and jitter

**Backoff** protects the dependency: each retry waits longer than the last, giving it room to recover.
**Jitter** protects the recovery: without randomness, every client that failed together retries together.

In [ ]:
def backoff_delay(attempt: int, base=0.5, cap=8.0, jitter="full", rng=random) -> float:
    d = min(cap, base * 2 ** attempt)                          # the exponential ceiling
    if jitter == "none":
        return d
    if jitter == "full":
        return rng.uniform(0, d)                               # anywhere in [0, ceiling]
    if jitter == "equal":
        return d / 2 + rng.uniform(0, d / 2)                   # keep half, randomise half
    raise ValueError(jitter)

LABEL = {"none": "no jitter", "equal": "equal jitter", "full": "full jitter"}
rng = random.Random(0)
attempts = np.arange(8)
fig, ax = plt.subplots(figsize=(9, 3.8))
for off, (kind, color) in zip([-0.22, 0, 0.22], [("none", "#e03131"), ("equal", "#f08c00"), ("full", "#2f9e44")]):
    for a in attempts:
        ys = [backoff_delay(a, jitter=kind, rng=rng) for _ in range(150)]
        ax.scatter(np.full(len(ys), a + off) + rng.uniform(-0.05, 0.05), ys, s=30 if kind == "none" else 6,
                   alpha=0.35, color=color, label=LABEL[kind] if a == 0 else None)
ax.plot(attempts, [min(8.0, 0.5 * 2 ** a) for a in attempts], "k--", lw=1, label="ceiling = min(cap, base x 2^n)")
ax.set(xlabel="retry attempt n", ylabel="wait before retrying (s)",
       title="Backoff schedules for 150 clients: the ceiling doubles, jitter decides where below it each one waits")
ax.legend(loc="upper left"); plt.show()

With no jitter all 150 clients sit on one point per attempt. **Full jitter** (uniform between 0 and the ceiling,
from the AWS "Exponential Backoff and Jitter" analysis) spreads them the most; equal jitter keeps a guaranteed
minimum wait. Two caps matter as much as the growth: maximum attempts and maximum total wall-clock time.

## 4. Why jitter is not optional: a thundering-herd simulation

100 clients all fail at the same instant (a replica restarts, a quota window resets). The server can serve 10
requests per 50 ms window; extra requests in a window are rejected and those clients back off again.

In [ ]:
def simulate_herd(jitter, n_clients=100, capacity=10, window=0.05, base=0.1, cap=5.0, max_attempts=15, seed=0):
    rng = random.Random(seed)
    queue = [(backoff_delay(0, base, cap, jitter, rng), i, 1) for i in range(n_clients)]   # (time, client, attempt)
    heapq.heapify(queue)
    served, sends, done_at = {}, [], []
    while queue:
        t, i, attempt = heapq.heappop(queue)
        sends.append(t)
        w = int(t // window)
        if served.get(w, 0) < capacity:                    # room in this window: success
            served[w] = served.get(w, 0) + 1
            done_at.append(t)
        elif attempt < max_attempts:                       # rejected: back off and try again
            heapq.heappush(queue, (t + backoff_delay(attempt, base, cap, jitter, rng), i, attempt + 1))
    return np.array(sends), np.sort(done_at)

herd = {kind: simulate_herd(kind) for kind in ("none", "equal", "full")}
for kind, (sends, done) in herd.items():
    peak = np.bincount((sends // 0.05).astype(int)).max()
    print(f"{LABEL[kind]:>12}: {len(sends):4d} requests sent for 100 successes, peak {peak:3d} per window, "
          f"last client done at {done[-1]:5.2f} s")
assert len(herd["full"][0]) < len(herd["none"][0]) / 2, "full jitter does far less wasted work"
assert herd["full"][1][-1] < herd["none"][1][-1] / 5, "and finishes far sooner"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
bins = np.arange(0, 2.0001, 0.05)
for kind, color in [("none", "#e03131"), ("full", "#2f9e44")]:
    axes[0].hist(herd[kind][0], bins=bins, histtype="step", lw=1.8, color=color, label=LABEL[kind])
axes[0].axhline(10, color="k", ls="--", lw=1, label="server capacity per window")
axes[0].set(xlabel="time (s)", ylabel="requests per 50 ms window", title="Load on the recovering server (first 2 s)")
axes[0].legend()
for kind, color in [("none", "#e03131"), ("equal", "#f08c00"), ("full", "#2f9e44")]:
    done = herd[kind][1]
    axes[1].step(done, np.arange(1, len(done) + 1), where="post", color=color, label=LABEL[kind])
axes[1].set_xscale("log")
axes[1].set(xlabel="time (s, log scale)", ylabel="clients that succeeded", title="Everyone done: full jitter wins")
axes[1].legend(); plt.tight_layout(); plt.show()

Without jitter every retry wave arrives in **one** window: 100, then 90, then 80... The server serves 10 each time
and rejects the rest, so the herd needs ten waves: 550 requests for 100 successes and 26 s. Jitter turns each wave
into a trickle the server can absorb. This is the picture behind "backoff protects the dependency, jitter
protects the recovery" (`po08`).

## 5. A retry loop that honours `Retry-After` and a deadline

When the server says how long to wait, that is not advice: rate limits are exactly the case where it knows
better than you. Otherwise use full jitter. And never wait past the request's deadline: a retry nobody is waiting
for is pure cost.

In [ ]:
def retry_after_seconds(exc) -> float | None:
    """Server-directed wait: `retry-after-ms` (OpenAI), else `retry-after` as seconds or an HTTP date."""
    headers = getattr(getattr(exc, "response", None), "headers", None) or {}
    if headers.get("retry-after-ms"):
        return float(headers["retry-after-ms"]) / 1000
    value = headers.get("retry-after")
    if value is None:
        return None
    try:
        return float(value)
    except ValueError:
        return max(0.0, email.utils.parsedate_to_datetime(value).timestamp() - time.time())

def call_with_retries(fn, *, max_attempts=4, base=0.2, cap=5.0, deadline_s=10.0, rng=random.Random(0), log=print):
    start = time.monotonic()
    for attempt in range(max_attempts):
        try:
            return fn()
        except Exception as exc:
            name = type(exc).__name__
            if not is_retryable(exc) or attempt == max_attempts - 1:
                log(f"  attempt {attempt + 1}: {name} -> {'not retryable' if not is_retryable(exc) else 'out of attempts'}, raising")
                raise
            server_wait = retry_after_seconds(exc)
            wait = server_wait if server_wait is not None else backoff_delay(attempt, base, cap, "full", rng)
            if time.monotonic() - start + wait > deadline_s:
                log(f"  attempt {attempt + 1}: {name} -> a {wait:.2f} s wait would miss the deadline, raising")
                raise
            log(f"  attempt {attempt + 1}: {name} -> sleep {wait:.2f} s ({'server said so' if server_wait is not None else 'full jitter'})")
            time.sleep(wait)

First with a simulated network blip (no headers, so full jitter), which runs the same with any provider; then,
offline, real SDK errors: two 429s that carry `retry-after-ms`, and a 400 that must not be retried.

In [ ]:
def flaky(n_failures, result="ok"):
    """A call that drops the connection n times, then works."""
    state = {"left": n_failures}
    def fn():
        if state["left"] > 0:
            state["left"] -= 1
            raise openai.APIConnectionError(request=req)
        return result
    return fn

print("network blips:");      print("  ->", call_with_retries(flaky(2)))
ask = lambda: c0.chat.completions.create(model=MODEL, messages=MSG, max_tokens=20).choices[0].message.content
if OFFLINE:
    OFFLINE.fail_next(2, 429)
    print("rate limited:");   print("  ->", call_with_retries(ask))
    OFFLINE.fail_next(1, 400)
    before = requests_served()
    try:
        print("bad request:"); call_with_retries(ask)
    except openai.BadRequestError:
        assert requests_served() - before == 1, "a 400 is sent exactly once"

### Retry amplification: retry at one layer only

The SDK retries, your loop retries, the gateway in front of you retries... Against a hard-down upstream those
multiply. Offline: our 3-attempt loop around the **default** client (which makes 3 HTTP attempts per call).

In [ ]:
if OFFLINE:
    amp = make_client(max_retries=2)             # the SDK default
    OFFLINE.fail_next(9, 503)                    # upstream is down for the next 9 requests
    before = requests_served()
    try:
        call_with_retries(lambda: amp.chat.completions.create(model=MODEL, messages=MSG), max_attempts=3,
                          log=lambda m: None)
    except openai.InternalServerError:
        pass
    n = requests_served() - before
    print(f"1 user request -> {n} HTTP requests to a service that is already down")
    assert n == 9

Three layers of 3 attempts is 27 requests per user request; at 10x traffic in an incident that is 270x load on
the thing that is failing (`po21`). Retry at **one** layer (the outermost one that knows the deadline) and set
`max_retries=0` below it. At fleet level add a **retry budget**: allow retries only up to, say, 10% of recent
successful traffic, so a broad outage cannot multiply load.

## 6. The same policy with `tenacity`

`tenacity` is the standard Python retry library (it is already in this venv as a dependency of LangChain). It
expresses the policy declaratively. It does not read `Retry-After` by itself, but `wait=` accepts any function,
so we plug ours in.

In [ ]:
from tenacity import retry, retry_if_exception, stop_after_attempt, stop_after_delay, wait_random_exponential

jitter_wait = wait_random_exponential(multiplier=0.2, max=5.0)     # full jitter: uniform(0, min(5, 0.2 x 2^n))

def wait_server_or_jitter(retry_state) -> float:
    ra = retry_after_seconds(retry_state.outcome.exception())
    return ra if ra is not None else jitter_wait(retry_state)

@retry(retry=retry_if_exception(is_retryable),
       wait=wait_server_or_jitter,
       stop=stop_after_attempt(4) | stop_after_delay(10),      # cap attempts AND total time
       reraise=True)                                           # raise the real error, not tenacity.RetryError
def robust_chat(prompt: str) -> str:
    r = c0.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=20)
    return r.choices[0].message.content

if OFFLINE:
    OFFLINE.fail_next(2, 503)
print("answer:", robust_chat("What is 12*12?"))
print("attempts used:", robust_chat.statistics["attempt_number"], "| time spent sleeping:",
      f"{robust_chat.statistics['idle_for']:.2f} s")
if OFFLINE:
    assert robust_chat.statistics["attempt_number"] == 3

## 7. Idempotency: make retries safe for side effects

Retrying a chat completion costs money but changes nothing. Retrying a **tool call** (issue a refund, send an
email) changes the world twice. The nasty case: the refund *succeeded*, but the response was lost on the way back,
so the caller sees a timeout and retries. The fix lives at the tool boundary: an **idempotency key** derived from
the *intent*, which the service uses to deduplicate.

In [ ]:
class RefundService:
    """A fake downstream tool that deduplicates on an idempotency key when one is given."""
    def __init__(self):
        self.executed, self.by_key = [], {}
    def refund(self, order_id, amount, idempotency_key=None):
        if idempotency_key in self.by_key:
            return self.by_key[idempotency_key]                   # replay: return the original result
        result = {"refund_id": f"rf_{len(self.executed) + 1}", "order_id": order_id, "amount": amount}
        self.executed.append(result)                              # the side effect
        if idempotency_key:
            self.by_key[idempotency_key] = result
        return result

def lossy_call(fn, lose_first_response=True):
    """The request reaches the service, but the first response is lost: the caller sees a timeout."""
    state = {"lost": not lose_first_response}
    def wrapper():
        result = fn()                                             # runs on the server either way
        if not state["lost"]:
            state["lost"] = True
            raise openai.APITimeoutError(request=req)
        return result
    return wrapper

def intent_key(*parts) -> str:
    return hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:16]

quiet = lambda m: None
for label, make_key in [("no key", lambda: None), ("uuid4 per attempt (wrong)", lambda: uuid.uuid4().hex),
                        ("key from the intent", lambda: intent_key("order-42", 19.99, "damaged"))]:
    svc = RefundService()
    call_with_retries(lossy_call(lambda: svc.refund("order-42", 19.99, idempotency_key=make_key())), log=quiet)
    print(f"{label:<27} -> refunds actually issued: {len(svc.executed)}")
    assert len(svc.executed) == (1 if label == "key from the intent" else 2)

The key must come from the **action** (order id, amount, reason), not the attempt: a fresh uuid per attempt makes
every retry look new. Keep keys for longer than your whole retry budget. Hedged requests (notebook 01) and queues
that deliver at least once need exactly the same property.

## 8. Rate-limit yourself: a token bucket

**In plain English:** a bucket holds up to `capacity` permits and refills at `rate` per second. Each request takes
one; an empty bucket means wait or reject. Capacity sets the burst you tolerate, rate sets the sustained ceiling.
This is what providers do to you, and what your gateway should do per tenant before anything expensive runs.

In [ ]:
class TokenBucket:
    def __init__(self, capacity: float, rate: float, now: float = 0.0):
        self.capacity, self.rate, self.tokens, self.t = capacity, rate, capacity, now
    def _refill(self, now):
        self.tokens = min(self.capacity, self.tokens + (now - self.t) * self.rate)
        self.t = now
    def allow(self, now: float, cost: float = 1.0) -> bool:
        self._refill(now)
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False
    def retry_after(self, cost: float = 1.0) -> float:
        """Seconds until `cost` permits will be available: send this back as Retry-After."""
        return float("inf") if cost > self.capacity else max(0.0, (cost - self.tokens) / self.rate)

rng_np = np.random.default_rng(0)
def poisson_arrivals(rate, t0, t1):
    n = rng_np.poisson(rate * (t1 - t0))
    return np.sort(rng_np.uniform(t0, t1, n))
arrivals = np.sort(np.concatenate([poisson_arrivals(4, 0, 20), poisson_arrivals(25, 5, 7), poisson_arrivals(40, 12, 13)]))

bucket = TokenBucket(capacity=10, rate=6)
allowed, levels = [], []
for t in arrivals:
    allowed.append(bucket.allow(t)); levels.append(bucket.tokens)
allowed = np.array(allowed)
print(f"{len(arrivals)} requests: {allowed.sum()} allowed, {(~allowed).sum()} rejected with 429")
ok_times = arrivals[allowed]
worst = max(((ok_times >= s) & (ok_times < s + 1)).sum() for s in np.arange(0, 20, 0.05))
print(f"most requests allowed in any 1 s window: {worst} (bound: capacity + rate x 1 s = 16)")
assert worst <= 10 + 6 * 1

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
bins = np.arange(0, 20.5, 0.5)
axes[0].hist([arrivals[allowed], arrivals[~allowed]], bins=bins, stacked=True, color=["#2f9e44", "#e03131"],
             label=["allowed", "rejected (429)"])
axes[0].axhline(6 * 0.5, color="k", ls="--", lw=1, label="sustained rate (6/s = 3 per bin)")
axes[0].set(ylabel="requests per 0.5 s", title="Token bucket (capacity 10, refill 6/s): bursts absorbed, floods clipped")
axes[0].legend(loc="upper left")
axes[1].step(arrivals, levels, where="post", color="#3b5bdb")
axes[1].set(xlabel="time (s)", ylabel="tokens left")
plt.tight_layout(); plt.show()

The first burst (5-7 s) is partly absorbed by the saved-up tokens; the flood at 12 s is clipped to the refill
rate. Production details that matter more than the algorithm:

- **Two buckets**, requests *and* tokens (RPM and TPM), because one 100k-token prompt costs more than a thousand
  short ones. Charge the TPM bucket `prompt_tokens + max_tokens` up front.
- **Per tenant and per API key**, enforced at the gateway, before retrieval or generation has spent anything.
- **Shared state** (a couple of fields in Redis): a per-replica limiter of 100/min becomes 1,000/min with ten pods.
- Return **429 with `Retry-After`** (from `retry_after()`), or your clients will retry blind.

In [ ]:
tpm = TokenBucket(capacity=30_000, rate=30_000 / 60)         # 30k tokens per minute
for cost in (2_000, 25_000, 40_000):
    ok = tpm.allow(now=0.0, cost=cost)
    wait = tpm.retry_after(cost)
    verdict = "allowed" if ok else ("reject with 413/400: can never fit" if wait == float("inf") else f"429, Retry-After {wait:.0f} s")
    print(f"request costing {cost:>6,} tokens -> {verdict}")

A request bigger than the whole bucket must be rejected as *non-retryable* (too large), not with a 429 that
invites a retry that can never succeed.

## 9. Circuit breaker: stop calling a dead service

Retries handle one unlucky call; a **circuit breaker** stops a thousand unlucky calls from becoming your outage.
Three states:

```
            failure rate >= threshold               cool-down elapsed
  CLOSED  ---------------------------->  OPEN  ------------------------>  HALF-OPEN
 (calls flow,                         (fail fast,                        (let one probe
  count results)                       use fallback)                      through)
     ^                                     ^                                  |
     |          probe succeeds             |          probe fails             |
     +-------------------------------------+----------------------------------+
```

In [ ]:
class CircuitBreaker:
    """Sequential-use sketch. A concurrent version needs a lock and a single probe slot."""
    def __init__(self, threshold=0.5, window=10, min_calls=5, cooldown=5.0):
        self.threshold, self.min_calls, self.cooldown = threshold, min_calls, cooldown
        self.results, self.state, self.opened_at = deque(maxlen=window), "closed", None
    def allow(self, now: float) -> bool:
        if self.state == "open" and now - self.opened_at >= self.cooldown:
            self.state = "half_open"                      # let one probe through
        return self.state != "open"
    def record(self, ok: bool, now: float):
        if self.state == "half_open":                     # the probe decides
            self.state, self.opened_at = ("closed", None) if ok else ("open", now)
            self.results.clear()
            return
        self.results.append(ok)
        fails = self.results.count(False)
        if len(self.results) >= self.min_calls and fails / len(self.results) >= self.threshold:
            self.state, self.opened_at = "open", now

def failure_prob(t):          # the flaky upstream: an outage from 15 s to 35 s, shaky until 40 s
    return 1.0 if 15 <= t < 35 else 0.3 if 35 <= t < 40 else 0.02

def simulate_breaker(use_breaker=True, cooldown=5.0, dt=0.5, T=60, seed=0):
    ts = np.arange(0, T, dt)
    u = np.random.default_rng(seed).random(len(ts))     # same luck for both runs
    cb, log = CircuitBreaker(cooldown=cooldown), []
    for t, x in zip(ts, u):
        if use_breaker and not cb.allow(t):
            log.append((t, "short-circuited", "open")); continue
        state = cb.state                               # "closed", or "half_open" for a probe
        ok = x >= failure_prob(t)
        if use_breaker:
            cb.record(ok, t)
        log.append((t, "success" if ok else "failure", state))
    return log

with_cb, without_cb = simulate_breaker(True), simulate_breaker(False)
def hits_on_dead_service(log):
    return sum(1 for t, o, _ in log if 15 <= t < 35 and o != "short-circuited")
print(f"calls sent to the dead service during the outage: without breaker {hits_on_dead_service(without_cb)}, "
      f"with breaker {hits_on_dead_service(with_cb)}")
print("outcomes with breaker:", {o: sum(1 for _, x, _ in with_cb if x == o) for o in ("success", "failure", "short-circuited")})
assert hits_on_dead_service(with_cb) < hits_on_dead_service(without_cb) / 3

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
ts = [t for t, _, _ in with_cb]
level = {"closed": 0, "half_open": 1, "open": 2}
axes[0].step(ts, [level[s] for *_, s in with_cb], where="post", color="#3b5bdb")
axes[0].set_yticks([0, 1, 2], ["closed", "half-open", "open"])
axes[0].set_title("Circuit breaker around a flaky upstream (outage shaded)")
style = {"success": ("o", "#2f9e44", 2), "failure": ("x", "#e03131", 1), "short-circuited": ("s", "#adb5bd", 0)}
for outcome, (marker, color, y) in style.items():
    xs = [t for t, o, _ in with_cb if o == outcome]
    axes[1].scatter(xs, [y] * len(xs), marker=marker, color=color, s=18, label=outcome)
axes[1].set_yticks([0, 1, 2], ["short-circuited\n(fallback)", "failure", "success"])
axes[1].set_xlabel("time (s)")
for ax in axes:
    ax.axvspan(15, 35, color="#ffe3e3", zorder=0)
plt.tight_layout(); plt.show()

Reading the timeline: a few failures trip the breaker, then calls fail fast (grey) and go to the fallback without
touching the upstream; every 5 s a single half-open probe checks for recovery; the first successful probe closes
it. Put **one breaker per dependency** (each provider, the vector DB, each tool), never one global breaker. And an
open breaker needs a fallback behind it, or you have only made your failures faster.

## 10. Fallback to a secondary model or provider

A fallback chain tries tiers in order and falls through only on errors the next tier can fix: a 5xx, timeout or
sustained 429 means "try the backup"; a 400 means your request is malformed and the backup will reject it
identically. Mark the answer as degraded, and never cache it.

In [ ]:
PRIMARY = make_client(max_retries=1)
SECONDARY = make_client(max_retries=1)   # in production: another provider or region (other base_url, key, model)
TIERS = [("primary", PRIMARY, MODEL), ("secondary", SECONDARY, MODEL)]

def chat_with_fallback(messages):
    for name, c, model in TIERS:
        try:
            r = c.chat.completions.create(model=model, messages=messages, max_tokens=30)
            return {"text": r.choices[0].message.content, "served_by": name, "degraded": name != "primary"}
        except (openai.RateLimitError, openai.InternalServerError, openai.APIConnectionError) as e:
            print(f"  {name} failed with {type(e).__name__}, falling through")
    return {"text": "The assistant is unavailable right now; here are the most relevant help articles.",
            "served_by": "canned", "degraded": True}

if OFFLINE:
    OFFLINE.fail_next(2, 503)                  # the primary's attempt and its one retry both fail
print(chat_with_fallback(MSG))
print(chat_with_fallback(MSG))                 # primary healthy again
if OFFLINE:
    OFFLINE.fail_next(1, 400)
    try:
        chat_with_fallback(MSG)
    except openai.BadRequestError:
        print("a 400 is not failed over: the backup would reject the same request")

Requirements people forget (`in28`, `po03`): prompts that port across providers (templates, tool-call formats and
structured-output support differ), an eval of the fallback so you know what quality you accept, the model id
logged on every response, and a small **continuous** slice of real traffic through the fallback so it is known to
work before the incident. [LiteLLM's Router](06_litellm_gateway_and_model_routing.ipynb) gives you retries,
fallbacks and per-deployment cooldowns (a breaker) as configuration.

## Try it yourself

**1.** One noisy tenant sends 20 req/s, a normal tenant 2 req/s. Compare a single global bucket (capacity 10,
10/s) with per-tenant buckets (capacity 5, 5/s each): what share of the *normal* tenant's requests get through?

In [ ]:
# Solution — try it yourself first, then run this
r2 = np.random.default_rng(1)
events = sorted([(t, "noisy") for t in r2.uniform(0, 10, 200)] + [(t, "normal") for t in r2.uniform(0, 10, 20)])
global_bucket = TokenBucket(10, 10)
per_tenant = {"noisy": TokenBucket(5, 5), "normal": TokenBucket(5, 5)}
g_ok, p_ok = {"noisy": 0, "normal": 0}, {"noisy": 0, "normal": 0}
for t, who in events:                                 # both limiters see the same traffic
    g_ok[who] += global_bucket.allow(t)
    p_ok[who] += per_tenant[who].allow(t)
print(f"normal tenant success: global bucket {g_ok['normal']}/20, per-tenant buckets {p_ok['normal']}/20")
assert p_ok["normal"] == 20 and g_ok["normal"] < 20

**2.** Implement a **retry budget**: retries are allowed only while `retries <= 0.1 x successes` over the recent
past. During a total outage of 1,000 requests with a 3-retry policy, how many upstream calls do you make with and
without the budget? (Assume the budget starts with credit for 100 earlier successes.)

In [ ]:
# Solution — try it yourself first, then run this
class RetryBudget:
    def __init__(self, ratio=0.1, prior_successes=100):
        self.ratio, self.successes, self.retries = ratio, prior_successes, 0
    def can_retry(self):
        return self.retries < self.ratio * self.successes
    def spend(self):
        self.retries += 1

def outage_calls(budget=None, requests=1000, retries_per_request=3):
    calls = 0
    for _ in range(requests):
        calls += 1                                     # first attempt fails
        for _ in range(retries_per_request):
            if budget is not None and not budget.can_retry():
                break
            if budget is not None:
                budget.spend()
            calls += 1                                 # each retry also fails
    return calls

print(f"no budget: {outage_calls()} calls, with a 10% budget: {outage_calls(RetryBudget())} calls")
assert outage_calls(RetryBudget()) == 1010

**3.** Re-run the breaker simulation with a 1 s and a 10 s cool-down. How many calls reach the dead service, and
how long after recovery (t = 40 s) does the breaker close?

In [ ]:
# Solution — try it yourself first, then run this
for cd in (1.0, 5.0, 10.0):
    log = simulate_breaker(True, cooldown=cd)
    closed_after = min((t for t, _, s in log if t >= 35 and s == "closed"), default=None)
    print(f"cool-down {cd:>4} s: {hits_on_dead_service(log):2d} calls hit the dead service, closed again at t={closed_after}")

if OFFLINE:                     # leave the stand-in clean
    OFFLINE.latency = 0.0

## Say it in an interview

- **30 seconds on 429s:** "Classify first: 429, 5xx, timeouts and connection errors are retryable; 400, 401, 404
  and content-policy errors are not. For 429 I honour `Retry-After`, otherwise exponential backoff with full
  jitter, capped on attempts *and* total time inside the request deadline. Retries happen at one layer only, with
  a fleet-level retry budget. A persistent 429 is a capacity signal: a token bucket per tenant sized to the quota,
  a queue in front, and a fallback provider rather than more waiting."
- **Backoff vs jitter:** backoff protects the dependency, jitter protects the recovery. In the herd simulation,
  no jitter sent more than twice the requests of full jitter and took ~25x as long to drain.
- **Circuit breaker:** closed, open, half-open; one per dependency; trips on a failure *rate* over a window; an
  open breaker must route to a fallback.
- **Idempotency:** retries are only safe for side effects when the tool deduplicates on a key derived from the
  intent, not from the attempt.
- **Traps:** a blanket retry decorator on everything; retrying 400s; SDK retries + your retries + gateway retries
  (27x); per-replica rate limiters; rate limiting by request count only (limit tokens and money); caching a
  degraded fallback answer; a fallback nobody has exercised in six months.

## Next

- [03 · Caching, exact and semantic](03_caching_exact_and_semantic.ipynb): the cheapest request is the one you never send.
- [06 · LiteLLM gateway and model routing](06_litellm_gateway_and_model_routing.ipynb): retries, fallbacks and cooldowns as config.
- [08 · Observability](08_observability_logging_and_tracing.ipynb): alert on 429 rate and breaker state, not single errors.
- Related: [01 · Async concurrency](01_async_concurrency_for_llm_calls.ipynb) · [agent guardrails and failure modes](../15_agentic_ai/06_guardrails_hitl_and_failure_modes.ipynb).
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html).